# The joint manifold of session time and reward time (LEC)

Tests the hypotheses in `mohamady_time_idea/time_torus1.pdf` and `time_torus2.pdf`.
Companion documentation: [`TIME_MANIFOLD.md`](TIME_MANIFOLD.md). Module:
[`time_manifold.py`](time_manifold.py).

We have two population time signals — time-in-session `T` and time-since-reward
`tau`. This notebook asks what **joint geometry** they make: an open sheet, a
cylinder, or a torus.

**Read section 4 before section 6.** Report 1's central point is that topology
cannot answer the interesting question: separable, sheared, warped and conjunctive
codes are *all* beta1 = 0 sheets with completely different computational content.
Only cross-condition generalisation separates them.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from importlib import reload
import time_manifold as tm; reload(tm)
import glm_analysis_v2 as glm
glm.apply_gridmaze_style()

# Kernel guard. `ripser` lives in the `maze_ephys` env, NOT `maze_ephys_si104`.
# Run this notebook under maze_ephys. Without ripser every H1 in the gate reads
# nan, `_sheet_ceiling` has nothing to compare against, and the topology checks
# report FAILED for a reason that has nothing to do with the data.
import ripser  # noqa: F401

SAVE_DIR = '../data/time_manifold_outputs/LEC'
SAVE_FIGS = True

In [2]:
# Load the LEC data dictionary
import os
import glm_analysis_v2 as glm

processed = "../data/processed_data"

# load_data_dic drops known-bad recdays and asserts that each recday's unit count
# and task sequences come from the recording day it is named after -- see
# code/recday_registry.py and docs/BUG_ly05_recday_mismatch.md. A bare
# pickle.load cannot catch a recday paired to the wrong day's sorting.
data_dic = glm.load_data_dic(os.path.join(processed, 'data_dic_lec.pkl'))
mouse_recdays = sorted([r for r in data_dic if '_sb' not in str(r)])
print(len(mouse_recdays), 'recdays')

24 recdays


## 1. Substrate

`build_time_tables` adds `T_sec` — elapsed seconds since session start, from the
raw bin index. This variable does not exist anywhere else in the repo:
`glm_analysis_v2` has no session-time regressor and
`time_coding_analysis.calculate_time_variables` computes it as `trial_index * 360 / 40`
on the phase-warped substrate, which is trial index, not seconds.

In [3]:
cfg = tm.TimeManifoldConfig()
tables = tm.build_time_tables(mouse_recdays, data_dic, cfg)
print(f'\n{len(tables)} recdays with usable tables')

  ah08_20250613_20250615: 151 neurons x 13664 samples, 6 sessions, 224 legs, 63 loops
  ah08_20250616_20250617: 164 neurons x 18434 samples, 5 sessions, 280 legs, 73 loops
  ah08_20250618_20250619: 183 neurons x 23637 samples, 6 sessions, 384 legs, 99 loops
  ah08_20250620_20250623: 108 neurons x 21102 samples, 6 sessions, 386 legs, 99 loops
  ah08_20250624_20250625: 186 neurons x 9496 samples, 4 sessions, 127 legs, 36 loops
  ah10_20250613_20250615: 139 neurons x 28714 samples, 6 sessions, 706 legs, 177 loops
  ah10_20250616_20250617: 154 neurons x 29619 samples, 6 sessions, 616 legs, 154 loops
  ah10_20250618_20250619: 162 neurons x 23281 samples, 5 sessions, 570 legs, 143 loops
  ah10_20250620_20250623: 153 neurons x 27698 samples, 6 sessions, 747 legs, 187 loops
  ah10_20250624_20250625: 164 neurons x 27615 samples, 6 sessions, 664 legs, 166 loops
  ly05_20250613_20250615: 117 neurons x 18013 samples, 6 sessions, 305 legs, 80 loops
  ly05_20250616_20250617: 116 neurons x 16550 samp

In [4]:
# Leg-duration leverage and the within-session duration drift.
# The drift is why the transfer test in section 4 must be range-matched
# (TIME_MANIFOLD.md 8.1) -- without it a separable code is indistinguishable
# from a conjunctive one.
from scipy.stats import spearmanr
for rd, t in tables.items():
    s = tm.leg_duration_stats(t)
    _, first = np.unique(t['interval_id'], return_index=True)
    r, p = spearmanr(t['T_frac'][first], t['D'][first])
    print(f"{rd:26s} legs={s['n']:4d} p10={s['p10']:5.1f} med={s['median']:5.1f} "
          f"p90={s['p90']:5.1f} (x{s['ratio']:.1f})  dur-vs-T rho={r:+.3f} p={p:.1e}")

ah08_20250613_20250615     legs= 224 p10=  6.0 med= 11.9 p90= 29.8 (x5.0)  dur-vs-T rho=-0.376 p=6.1e-09
ah08_20250616_20250617     legs= 280 p10=  7.1 med= 13.5 p90= 28.3 (x4.0)  dur-vs-T rho=-0.299 p=3.5e-07
ah08_20250618_20250619     legs= 384 p10=  5.9 med= 12.0 p90= 30.1 (x5.1)  dur-vs-T rho=-0.223 p=1.1e-05
ah08_20250620_20250623     legs= 386 p10=  5.2 med= 11.2 p90= 25.4 (x4.9)  dur-vs-T rho=-0.223 p=9.4e-06
ah08_20250624_20250625     legs= 127 p10=  5.2 med= 14.6 p90= 36.0 (x6.9)  dur-vs-T rho=-0.256 p=3.7e-03
ah10_20250613_20250615     legs= 706 p10=  4.2 med=  8.1 p90= 18.0 (x4.2)  dur-vs-T rho=-0.282 p=2.3e-14
ah10_20250616_20250617     legs= 616 p10=  5.5 med= 10.8 p90= 19.9 (x3.6)  dur-vs-T rho=-0.170 p=2.1e-05
ah10_20250618_20250619     legs= 570 p10=  4.5 med=  8.5 p90= 19.2 (x4.3)  dur-vs-T rho=-0.281 p=7.9e-12
ah10_20250620_20250623     legs= 747 p10=  3.9 med=  8.0 p90= 15.6 (x4.1)  dur-vs-T rho=-0.257 p=1.1e-12
ah10_20250624_20250625     legs= 664 p10=  4.6 med=  8.

## 2. THE GATE — run this before trusting any number below

Ten models with known ground truth, injected into a **real** table (real leg
boundaries, real durations, real session structure, real occupancy, real N) and
driven through the **real** functions unmodified.

The checks are contrasts between models, not thresholds on absolute values — our
timescales are not report 1's, so what has to survive the port is which model
scores higher than which, and a gate tuned until it passes is not a gate.

In [5]:
gate = tm.run_synthetic_controls(list(tables.values())[0], cfg)
assert gate['passed'], 'GATE FAILED — do not interpret anything below'

KeyboardInterrupt: 

In [ ]:
# The gate is per-template. Run it on every recday you intend to analyse:
# `min_loops_for_loop_topology` means the loop-phase axis is only asserted on
# recdays with enough A->A loops (measured limit ~90; see TIME_MANIFOLD.md 9).
for rd, t in tables.items():
    g = tm.run_synthetic_controls(t, cfg, verbose=False)
    print(f"{rd:26s} {'PASS' if g['passed'] else 'FAIL'} "
          f"{sum(g['checks'].values())}/{len(g['checks'])}  "
          f"n_loops={g['n_loops']} loop-topology={'on' if g['loop_powered'] else 'underpowered'}")
    for k, v in g['checks'].items():
        if not v:
            print('      FAILED:', k)

## 3. The reset test — the headline

Population-vector correlation with the just-after-reward state as a function of
`tau`. **If the fast axis closes the curve comes back up; if it resets the curve
decays to a plateau and stays there.** No dimensionality reduction involved.

The plateau sits well above zero because a tiled code shares a mean — the
diagnostic is the *shape*, not the floor.

Both the trimmed and untrimmed curves are shown: `tau ~ 0` sits inside the reward
window, so a recovery present only in the untrimmed curve is a reward transient.

In [ ]:
for rd, t in list(tables.items())[:3]:
    axes = tm.plot_reset_curves(t, cfg)
    axes[0].figure.suptitle(rd, fontsize=8)
    axes[0].figure.tight_layout()
    if SAVE_FIGS: glm.save_section('reset_' + rd, SAVE_DIR)
    plt.show()

In [ ]:
# Population summary of the curve shape, per binning.
import pandas as pd
rows = []
for rd, t in tables.items():
    trimmed = tm.trim_mask(t, config=cfg)
    for b in tm.FAST_AXIS_BINNINGS:
        s = tm.return_curve_shape(tm.reset_return_curve(t, cfg, binning=b, mask=trimmed))
        rows.append(dict(recday=rd, binning=b, **s))
shape_df = pd.DataFrame(rows)
shape_df.groupby('binning')[['dip', 'recovery', 'corr_end']].describe().round(2)

## 4. Closure, and the binning that decides it

Report 1's most practical finding: **bin the fast axis every way, always.** A
phase-like code reads closure ~1.00 binned by phase and ~0.50 binned by absolute
seconds; a true sheet reads ~0.00 under both. One binning alone cannot tell them
apart.

The closure index reads the ends of the **analysis window**, not the topology of
the axis — the window is annotated on the x-axis for exactly that reason.

In [ ]:
for rd, t in list(tables.items())[:3]:
    ax, res = tm.plot_closure_summary(t, cfg)
    ax.figure.suptitle(rd, fontsize=8); ax.figure.tight_layout()
    if SAVE_FIGS: glm.save_section('closure_' + rd, SAVE_DIR)
    plt.show()

## 5. Factorisation — the test that actually separates the hypotheses

Topology cannot distinguish a separable product code from a conjunctive one; both
are sheets. Cross-condition generalisation can, and it is the property that
matters computationally.

**`tau_transfer_ratio` is the headline** — transfer divided by its own
within-condition ceiling, both computed on a range-matched window. The raw
transfer R2 means nothing without the ceiling beside it.

In [ ]:
fac = {rd: tm.factorisation(t, cfg, mask=tm.trim_mask(t, config=cfg))
       for rd, t in tables.items()}
for rd, f in fac.items():
    print(f"{rd:26s} N={f['n_neurons']:3d}  "
          f"tau: {f['tau_across_T']['r2']:+.2f}/{f['tau_within']['r2']:+.2f} "
          f"= {f['tau_transfer_ratio']:+.2f}   "
          f"T: {f['T_across_tau']['r2']:+.2f}/{f['T_within']['r2']:+.2f} "
          f"= {f['T_transfer_ratio']:+.2f}")
ax = tm.plot_factorisation(fac)
if SAVE_FIGS: glm.save_section('factorisation', SAVE_DIR)
plt.show()

In [ ]:
# Additive index: is the code f(T) + g(tau) or conjunctive?
# Cross-validated, so it is not biased by the code's SNR (TIME_MANIFOLD.md 8.2).
for rd, t in tables.items():
    a = tm.additive_r2(t, cfg, mask=tm.trim_mask(t, config=cfg))
    print(f"{rd:26s} additive_index={a['additive_index']:+.2f} "
          f"(ceiling r2_full={a['r2_full']:+.2f})")

In [ ]:
# Subspace geometry: are the axes orthogonal, and does the fast code drift?
# tau_drift_excess subtracts the interleaved-halves null -- a raw drift angle is
# nonzero even when the true drift is zero (TIME_MANIFOLD.md 8.3).
for rd, t in tables.items():
    g = tm.axis_geometry(tm.build_condition_tensor(t, cfg, tau_binning='abs'))
    print(f"{rd:26s} angle(T,tau)={g['angle_T_tau']:5.1f} deg   "
          f"drift={g['tau_drift']:5.1f} null={g['tau_drift_null']:5.1f} "
          f"excess={g['tau_drift_excess']:+5.1f}")

## 6. The cross-session reset — the strongest test for the slow axis

Not available in report 1's simulation, which has one session. We have 6–9 per
recday.

Report 1 §1: cross-session generalisation is precisely what makes the slow axis
*not* a loop — "the state at T recurs at the start of every session, which is a
jump back to the origin, not a lap around a loop". It is also the cleanest
available separation of a genuine session-time code from electrode drift or
satiety, neither of which resets when a new session starts.

Sessions here are different **tasks** (`get_sessions_for_glm` dedups to one per
unique task), so this inherits the remapping caveat from
`remapping_rotation_analysis`.

In [ ]:
rows = []
for rd, t in tables.items():
    for r in tm.cross_session_transfer(t, cfg, mask=tm.trim_mask(t, config=cfg)):
        rows.append(dict(recday=rd, **r))
xs = pd.DataFrame(rows)
print(xs.groupby('recday')[['tau_r2', 'T_r2']].mean().round(3))
print('\npooled median  tau_r2 = %.3f   T_r2 = %.3f'
      % (xs['tau_r2'].median(), xs['T_r2'].median()))

## 7. Topology — last, and never from a single run

Two preprocessing choices decide whether ground-truth topology is recovered at all,
and both failures look like clean negative results: the metric must be
kNN-geodesic (Euclidean saturates on a tiled code) and the manifold must be
resampled uniformly in **neural arclength**, not in seconds.

**Never report a single-run beta1.** Report 1 detects a spurious ring in 4/6 runs
at 25 cells and 0/6 at 400: the point estimate looks like a ring and the
run-to-run spread is as large as the effect. `h1_stability` resamples neurons and
behavioural units; the spread is the result.

In [ ]:
for rd, t in list(tables.items())[:3]:
    stab = {b: tm.h1_stability(t, cfg, tau_binning=b, axis='tau', n_runs=8)
            for b in tm.FAST_AXIS_BINNINGS}
    for b, s in stab.items():
        print(f"{rd:26s} {b:11s} H1={s['H1_mean']:5.2f} +- {s['H1_sd']:4.2f} "
              f"detected {s['detected']}/{s['n_runs']} unit={s['unit']} "
              f"stable_ring={s['stable_ring']}")
    ax = tm.plot_h1_stability(stab, config=cfg)
    ax.figure.suptitle(rd, fontsize=8); ax.figure.tight_layout()
    if SAVE_FIGS: glm.save_section('h1_' + rd, SAVE_DIR)
    plt.show()
    print()

### 7b. The existing task-phase ring, re-binned in absolute time

`persistent_homology_analysis.analyse_taskphase_ring` runs on `Neurons_norm`,
which is phase-warped by construction (90 bins/state x 4). That is report 1's
"binned by trial phase" case. The companion measurement is the same population
binned in absolute seconds — `tau_binning='abs'` above. A ring present under phase
binning and absent under absolute binning is a statement about the warp; a ring
present under both is a statement about the neural code.

## 8. Compression (report 2)

Report 2 withdrew its own torus claim after an adversarial audit — gridness,
module structure, the period-vs-field-width law and beta1 from folded data were
all reproduced by nulls containing no compression. Only three statistics survived,
and only those three are implemented.

**The rule: if you fold, fold the null too.** Any statistic conditioning on an
estimated period, phase or module assignment needs its null passed through the
same conditioning.

In [ ]:
# 8a. Covariance eigenspectrum: near-degenerate consecutive pairs are the
# linear-algebra statement of "this axis closes", and no weight-vector null can
# fake them. Report 2: 1.026 band-pass vs 1.153 low-pass. Report the whole
# spectrum -- the claim is a relative one.
for rd, t in list(tables.items())[:3]:
    for v in ('tau', 'T'):
        s = tm.covariance_spectrum(t, cfg, variable=v,
                                   mask=tm.trim_mask(t, config=cfg))
        if s.get('eigenvalues') is not None:
            print(f"{rd:26s} {v:4s} pair ratios "
                  + ' '.join(f'{x:.3f}' for x in s['pair_ratios']))

In [ ]:
# 8b. The 2-D lattice, and whether we could even see it.
# Report 2 predicts a period of ~19 min (T) and ~14 s (tau); both sit ABOVE the
# band this dataset can resolve, so a null result here is uninformative rather
# than a refutation.
for rd, t in list(tables.items())[:3]:
    b = tm.detectable_period_band(t, cfg)
    print(f"{rd:26s} detectable T period {b['T_band'][0]:.0f}-{b['T_band'][1]:.0f} s, "
          f"tau period {b['tau_band'][0]:.1f}-{b['tau_band'][1]:.1f} s")
print('\n' + tm.detectable_period_band(list(tables.values())[0], cfg)['note'])

## 9. What is deliberately not here

- the **jump-magnitude ratio** as evidence — report 1 §4 measures 1.15 to 3.37 for
  the same model depending on the baseline; `reset_jump` returns all three and is
  a figure panel only
- **gridness or module claims** without `random_nonneg_weight_null` on a
  bandwidth-matched basis
- **beta1 from folded data** without a folded null
- **participation ratio** as evidence about manifold dimension — PR ranges 2.0 to
  14.6 across report 1's models, every one of which is a 2-D manifold
- a hierarchy gradient fitted through two regions